# 🕌 Quran Video Generator - Colab

Generate stunning Quran video reels with Arabic text, English translations, and beautiful backgrounds.

## 🚀 Quick Start

1. Run this notebook
2. Click the Gradio link in output
3. Generate videos!

In [ ]:
# @title 📦 Install Requirements (Run this first!)
!pip install -q moviepy==1.0.3 pydub==0.25.1 requests==2.31.0 deep-translator==1.11.4 arabic-reshaper python-bidi numpy "Pillow==10.1" decorator==4.4.2 proglog==0.1.10 gradio==4.44.0 gdown
!pip install -q python-telegram-bot==20.7
print('✅ Dependencies installed! Restart runtime if prompted.')

In [ ]:
# @title 📥 Download Fonts from Google Drive
import gdown, shutil, os

FOLDER_ID = "1kSiHCRcznwUd1pU_pNaKdXx_y-hcRuIw"
DL_DIR    = "/content/drive_dl"
FONT_DIR  = "/content/fonts"

os.makedirs(DL_DIR, exist_ok=True)
os.makedirs(FONT_DIR, exist_ok=True)

gdown.download_folder(id=FOLDER_ID, output=DL_DIR, quiet=True, use_cookies=False)

for root, _, files in os.walk(DL_DIR):
    for fname in files:
        if fname.lower().endswith((".ttf", ".otf")):
            shutil.copy2(os.path.join(root, fname), os.path.join(FONT_DIR, fname))
            print(f'✅ {fname}')

In [ ]:
# @title ⚙️ Setup - Imports & Config
import os, time, json, requests
from io import BytesIO
from datetime import datetime

import gradio as gr
from pydub import AudioSegment
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeVideoClip, ColorClip, ImageSequenceClip

# Base paths
BASE_DIR = "/content/app"
FONT_DIR = "/content/fonts"
DATA_DIR = os.path.join(BASE_DIR, "data")
VIDEOS_DIR = os.path.join(DATA_DIR, "videos")
VISION_DIR = os.path.join(DATA_DIR, "vision")
TEMP_DIR = os.path.join(DATA_DIR, "temp")
CACHE_DIR = os.path.join(DATA_DIR, "cache")

for directory in [BASE_DIR, DATA_DIR, VIDEOS_DIR, VISION_DIR, TEMP_DIR, CACHE_DIR]:
    os.makedirs(directory, exist_ok=True)

# Fonts
FONT_ARABIC = os.path.join(FONT_DIR, "Arabic.ttf")
FONT_ENGLISH = os.path.join(FONT_DIR, "English.otf")

# Reciters
NEW_RECITERS = {
    "احمد النفيس": (259, "https://server16.mp3quran.net/nufais/Rewayat-Hafs-A-n-Assem/"),
    "وديع اليماني": (219, "https://server6.mp3quran.net/wdee3/"),
    "بندر بليلة": (217, "https://server6.mp3quran.net/balilah/"),
    "ادريس ابكر": (12, "https://server6.mp3quran.net/abkr/"),
    "منصور السالمي": (245, "https://server14.mp3quran.net/mansor/"),
    "رعد الكردي": (221, "https://server6.mp3quran.net/kurdi/"),
}

OLD_RECITERS = {
    "ابو بكر الشاطري": "Abu_Bakr_Ash-Shaatree_128kbps",
    "ياسر الدسري": "Yasser_Ad-Dussary_128kbps",
    "عبدالرحمن السديس": "Abdurrahmaan_As-Sudais_64kbps",
    "ماهر المعيقلي": "Maher_AlMuaiqly_64kbps",
    "سعود الشريم": "Saood_ash-Shuraym_64kbps",
    "مشاري العفاسي": "Alafasy_64kbps",
}

# Themes
THEMES = {
    "ذهبي": {"bg": (30, 20, 10), "text": (255, 215, 0), "translation": (200, 200, 200), "banner": (180, 140, 60)},
    "ليلي ازرق": {"bg": (10, 20, 40), "text": (255, 255, 255), "translation": (173, 216, 230), "banner": (25, 55, 90)},
    "اخضر": {"bg": (10, 30, 15), "text": (255, 255, 255), "translation": (144, 238, 144), "banner": (20, 60, 30)},
    "بنفسجي": {"bg": (40, 20, 50), "text": (255, 215, 0), "translation": (200, 180, 220), "banner": (60, 30, 80)},
    "ابيض": {"bg": (245, 245, 245), "text": (30, 30, 30), "translation": (80, 80, 80), "banner": (200, 200, 200)},
    "غروب": {"bg": (20, 10, 30), "text": (255, 255, 255), "translation": (255, 200, 150), "banner": (60, 30, 40)},
}

print('✅ Config loaded!')

In [ ]:
# @title 🔊 Audio & Text Functions
def get_verse_text(surah, ayah):
    """Get Arabic text and English translation"""
    ar_response = requests.get(f"https://api.alquran.cloud/v1/ayah/{surah}:{ayah}/quran-simple", timeout=10).json()
    arabic = ar_response["data"]["text"]
    
    en_response = requests.get(f"http://api.alquran.cloud/v1/ayah/{surah}:{ayah}/en.sahih", timeout=10).json()
    english = en_response["data"]["text"]
    
    return {"arabic": arabic, "english": english}

def get_audio_url(surah, reciter_id):
    """Get audio URL for a verse"""
    return f"https://server6.mp3quran.net/{reciter_id:03}/{surah:03}001.mp3"

def download_audio(url, path):
    """Download audio file"""
    response = requests.get(url, timeout=30)
    with open(path, "wb") as f:
        f.write(response.content)
    return path

def trim_silence(audio_path):
    """Trim silence from audio"""
    try:
        audio = AudioSegment.from_mp3(audio_path)
        audio = audio.strip_silence(silence_thresh=-40, padding=0)
        audio.export(audio_path, format="mp3")
    except: pass
    return audio_path

def get_arabic_text(text):
    """Reshape Arabic text for proper display"""
    reshaped = arabic_reshaper.reshape(text)
    return get_display(reshaped)

print('✅ Audio & text functions ready!')

In [ ]:
# @title 🎬 Video Generator
def create_video_frame(text_ar, text_en, theme, duration=5, width=720, height=1280):
    """Create a video frame with text"""
    # Create base image
    img = Image.new("RGBA", (width, height), theme["bg"] + (255,))
    draw = ImageDraw.Draw(img)
    
    # Load fonts (scaled)
    font_size = int(height / 15)
    try:
        font_ar = ImageFont.truetype(FONT_ARABIC, font_size)
        font_en = ImageFont.truetype(FONT_ENGLISH, font_size // 2)
    except:
        font_ar = ImageFont.load_default()
        font_en = ImageFont.load_default()
    
    # Remove Bismillah
    bismillah = "بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ"
    if text_ar.startswith(bismillah):
        text_ar = text_ar.replace(bismillah, "", 1)
    
    # Draw Arabic
    text_ar = get_arabic_text(text_ar)
    bbox = draw.textbbox((0, 0), text_ar, font=font_ar)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    x_ar = (width - text_w) // 2
    y_ar = height // 2 - 150
    draw.text((x_ar, y_ar), text_ar, font=font_ar, fill=theme["text"])
    
    # Draw English
    bbox_en = draw.textbbox((0, 0), text_en, font=font_en)
    text_en_w = bbox_en[2] - bbox_en[0]
    x_en = (width - text_en_w) // 2
    y_en = y_ar + text_h + 40
    draw.text((x_en, y_en), text_en, font=font_en, fill=theme["translation"])
    
    # Banner
    banner_h = 60
    draw.rectangle([(0, height - banner_h), (width, height)], fill=theme["banner"])
    
    return img

def generate_video(surah, ayah, reciter, reciter_id, theme_name, quality="720p"):
    """Generate a video for a verse"""
    # Get text
    text = get_verse_text(surah, ayah)
    
    # Get audio
    audio_url = get_audio_url(surah, reciter_id)
    audio_path = os.path.join(TEMP_DIR, f"audio_{surah}_{ayah}.mp3")
    download_audio(audio_url, audio_path)
    trim_silence(audio_path)
    
    # Dimensions
    dims = {"720p": (720, 1280), "1080p": (1080, 1920), "4K": (2160, 3840)}
    width, height = dims.get(quality, (720, 1280))
    
    # Create frame
    theme = THEMES.get(theme_name, THEMES["ذهبي"])
    frame = create_video_frame(text["arabic"], text["english"], theme, duration=5, width=width, height=height)
    
    # Load audio
    audio = AudioFileClip(audio_path)
    
    # Create video
    video = ImageSequenceClip([frame], duration=audio.duration)
    
    # Composite
    final = video.set_audio(audio)
    
    # Export
    output_path = os.path.join(VIDEOS_DIR, f"video_{surah}_{ayah}_{theme_name}.mp4")
    final.write_videofile(output_path, fps=24, codec="libx264", audio_codec="aac", verbose=False, logger=None)
    
    return output_path

print('✅ Video generator ready!')

In [ ]:
# @title 🎨 Gradio Interface (Run this!)
def generate(surah, ayah, reciter, theme, quality):
    """Generate video interface"""
    try:
        reciter_id = NEW_RECITERS.get(reciter, (12, ""))[0]
        video_path = generate_video(surah, ayah, reciter, reciter_id, theme, quality)
        return video_path
    except Exception as e:
        return f"Error: {str(e)}"

# Gradio interface
interface = gr.Interface(
    fn=generate,
    inputs=[
        gr.Slider(1, 114, step=1, label="Surah", value=1),
        gr.Slider(1, 286, step=1, label="Ayah", value=1),
        gr.Dropdown(list(NEW_RECITERS.keys()), label="Reciter", value="احمد النفيس"),
        gr.Dropdown(list(THEMES.keys()), label="Theme", value="ذهبي"),
        gr.Dropdown(["720p", "1080p", "4K"], label="Quality", value="720p"),
    ],
    outputs=gr.Video(label="Generated Video"),
    title="🕌 Quran Reel Maker",
    description="Generate beautiful Quran video reels"",
    examples=[
        [1, 1, "احمد النفيس", "ذهبي", "720p"],
        [1, 1, "بندر بليلة", "ليلي ازرق", "1080p"],
    ]
)

interface.launch(share=True, inline=False)

In [ ]:
# @title 📥 Download Generated Videos
import shutil

def list_videos():
    """List generated videos"""
    videos = []
    for f in os.listdir(VIDEOS_DIR):
        if f.endswith(".mp4"):
            videos.append(f)
    return videos

def download_video(filename):
    """Download a video"""
    path = os.path.join(VIDEOS_DIR, filename)
    if os.path.exists(path):
        return path
    return None

print("Generated videos:")
for v in list_videos():
    print(f"  - {v}")
print("\nDownload from Files panel (left)!")